In [8]:
import json
import sys

import pandas as pd

# notebooks run from notebooks/, so point Python one level up into ingestion/
sys.path.append("../ingestion")
from flatten_companyfacts import flatten

In [9]:
# Kraft Heinz: our known restatement case
with open("../data/raw/companyfacts/CIK0001637459.json") as f:
    raw = json.load(f)

df = flatten(raw)
df.shape

(22444, 14)

In [ ]:
df.head(10)

## Find value changes across filings

In [11]:
FACT_KEY = ["taxonomy", "concept", "unit", "start", "end"]


def find_changes(df):
    """Return one row per value change between consecutive filings of the same fact."""
    df = df.sort_values(FACT_KEY + ["filed", "accn"]).copy()
    by_fact = df.groupby(FACT_KEY, dropna=False)

    # shift(1) puts the previous filing's value next to each row, within the same fact
    df["prev_val"] = by_fact["val"].shift(1)
    df["prev_accn"] = by_fact["accn"].shift(1)
    df["prev_filed"] = by_fact["filed"].shift(1)
    df["prev_form"] = by_fact["form"].shift(1)

    # first report of a fact has no predecessor (prev_val is NaN) and is not a change
    changes = df[df["prev_val"].notna() & (df["val"] != df["prev_val"])].copy()

    changes["abs_change"] = changes["val"] - changes["prev_val"]
    # pct is undefined when the old value was 0
    changes["pct_change"] = changes["abs_change"] / changes["prev_val"].abs().replace(0, float("nan"))

    return changes[FACT_KEY + ["prev_val", "val", "abs_change", "pct_change",
                               "prev_accn", "accn", "prev_form", "form", "prev_filed", "filed"]]


changes = find_changes(df)
changes.shape

(817, 15)

### Check: Kraft Heinz FY2016 net income

In [12]:
khc_fy2016 = changes[(changes["concept"] == "NetIncomeLoss") & (changes["end"] == "2016-12-31")]
assert len(khc_fy2016) == 1, "known restatement not detected"
khc_fy2016

,taxonomy,concept,unit,start,end,prev_val,val,abs_change,pct_change,prev_accn,accn,prev_form,form,prev_filed,filed
12710,us-gaap,NetIncomeLoss,USD,2016-01-04,2016-12-31,3.632000e+09,3.596000e+09,-36000000.0,-0.009912,0001637459-18-000015,0001637459-19-000049,10-K,10-K,2018-02-16,2019-06-07


### How big are the changes?

In [13]:
print("changes:", len(changes))
print("\nabsolute % change, quantiles:")
print(changes["pct_change"].abs().quantile([0.1, 0.25, 0.5, 0.75, 0.9]).round(4))
print("\nconcepts that change most often:")
changes["concept"].value_counts().head(10)

changes: 817

absolute % change, quantiles:
0.10    0.0013
0.25    0.0053
0.50    0.0219
0.75    0.2000
0.90    1.4760
Name: pct_change, dtype: float64

concepts that change most often:


concept
OtherOperatingActivitiesCashFlowStatement               30
GrossProfit                                             21
OperatingIncomeLoss                                     21
SellingGeneralAndAdministrativeExpense                  21
OtherLiabilitiesCurrent                                 16
OtherNonoperatingIncomeExpense                          16
PaymentsForProceedsFromOtherInvestingActivities         16
ProceedsFromPaymentsForOtherFinancingActivities         16
IncreaseDecreaseInOtherCurrentAssets                    14
EffectiveIncomeTaxRateReconciliationOtherAdjustments    13
Name: count, dtype: int64

## Group changes by filing

In [ ]:
def revision_events(changes):
    """One row per filing that changed previously reported values."""
    return (
        changes.groupby(["accn", "form", "filed"])
        .agg(
            n_changes=("val", "size"),
            n_concepts=("concept", "nunique"),
            median_abs_pct=("pct_change", lambda s: s.abs().median()),
        )
        .sort_values("n_changes", ascending=False)
        .reset_index()
    )


khc_events = revision_events(changes)
print("filings with changes:", len(khc_events))
khc_events.head(5)

## Apple

In [ ]:
with open("../data/raw/companyfacts/CIK0000320193.json") as f:
    apple = flatten(json.load(f))

apple_changes = find_changes(apple)
apple_events = revision_events(apple_changes)
print("changes:", len(apple_changes), "| filings with changes:", len(apple_events))
apple_events.head(6)

### Stock splits

In [ ]:
def is_split_like(changes, tolerance=0.05):
    """True where a share-based value changed by a whole-number ratio (2x, 4x, 7x, ...)."""
    bigger = changes[["prev_val", "val"]].abs().max(axis=1)
    smaller = changes[["prev_val", "val"]].abs().min(axis=1)
    ratio = bigger / smaller.replace(0, float("nan"))

    share_based = changes["unit"].isin(["shares", "USD/shares"])
    whole_ratio = (ratio.round() >= 2) & ((ratio - ratio.round()).abs() < tolerance)
    return share_based & whole_ratio


apple_changes["split_like"] = is_split_like(apple_changes)
changes["split_like"] = is_split_like(changes)

print("Apple split-like changes:", apple_changes["split_like"].sum(), "of", len(apple_changes))
print("Kraft Heinz split-like changes:", changes["split_like"].sum(), "of", len(changes))
apple_changes[apple_changes["split_like"]].groupby("filed").size()

## Takeaways

Full write-up: [docs/findings](../docs/findings/)